In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

client = OpenAI()

In [7]:
import time

# abstract func to call llm api with telemetry manually logged
def call_llm(messages, model="gpt-4o", **kwargs):
    """
    Wraps the OpenAI call to return the text and core telemetry.
    """
    start_time = time.perf_counter()
    
    # Initialize telemetry with defaults
    telemetry = {
        "status": None,
        "latency_ms": 0,
        "tokens_prompt": 0,
        "tokens_completion": 0,
        "error": None
    }
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            **kwargs
        )
        
        # Calculate successful telemetry
        telemetry["model"] = model
        telemetry["latency_ms"] = round((time.perf_counter() - start_time) * 1000, 2)
        telemetry["status"] = 200
        telemetry["tokens_prompt"] = response.usage.prompt_tokens
        telemetry["tokens_completion"] = response.usage.completion_tokens
        
        return response.choices[0].message.content, telemetry

    except Exception as e:
        # Catch rate limits, timeouts, or auth errors
        telemetry["model"] = model
        telemetry["latency_ms"] = round((time.perf_counter() - start_time) * 1000, 2)
        telemetry["status"] = getattr(e, "status_code", "Error") 
        telemetry["error"] = str(e)
        
        return None, telemetry


In [11]:

def log_result(content, stats):
    """
    Prints a formatted summary of the LLM call with detailed token split.
    """
    total = stats['tokens_prompt'] + stats['tokens_completion']
    
    # 1. Print the Telemetry Bar
    # Format: [Model] Status: 200 | Time: 450ms | Tokens: 120 (In: 50 / Out: 70)
    print(f"[{stats['model']}] Status: {stats['status']} | Time: {stats['latency_ms']}ms")
    print(f"Tokens: {total} (In: {stats['tokens_prompt']} / Out: {stats['tokens_completion']})")
    
    # 2. Print Content or Error
    if content:
        # Using a separator for readability in dense notebooks
        print("-" * 20)
        print(f"Response: {content}")
        print("-" * 20 + "\n")
    elif stats['error']:
        print(f"⚠️ Error: {stats['error']}\n")


In [10]:
LLM_MODEL = "gpt-5-nano"

prompt = [{"role": "user", "content": "Tell me a 5-word space fact."}]
content, stats = call_llm(prompt, model= LLM_MODEL)
log_result(content, stats)

[gpt-5-nano] Status: 200 | Time: 8137.27ms | Tokens: 990
Response: Space is almost entirely vacuum.



In [12]:
import json
import time
import os

def call_llm_json(messages, model="gpt-4o", **kwargs):
    start_time = time.perf_counter()
    stats = {"model": model, "status": None, "latency_ms": 0, "tokens_prompt": 0, "tokens_completion": 0, "error": None}
    
    try:
        # response_format={"type": "json_object"} ensures the output is valid JSON
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            response_format={"type": "json_object"},
            **kwargs
        )
        
        stats.update({
            "status": 200,
            "latency_ms": round((time.perf_counter() - start_time) * 1000, 2),
            "tokens_prompt": response.usage.prompt_tokens,
            "tokens_completion": response.usage.completion_tokens
        })
        
        # Parse the string into a dict immediately so it's ready for saving
        content_dict = json.loads(response.choices[0].message.content)
        return content_dict, stats

    except Exception as e:
        stats["status"] = getattr(e, "status_code", "Error")
        stats["error"] = str(e)
        return None, stats

In [13]:
def save_single_json(data, filename="last_call.json"):
    """Saves the current output as a standalone file."""
    with open(filename, "w") as f:
        json.dump(data, f, indent=4)
    print(f"Saved snapshot to {filename}")

def append_to_master_json(data, stats, master_file="master_history.json"):
    """Appends the call and its telemetry to a list in a master file."""
    # Combine content and stats into one entry
    entry = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "telemetry": stats,
        "content": data
    }
    
    history = []
    if os.path.exists(master_file):
        with open(master_file, "r") as f:
            try:
                history = json.load(f)
            except json.JSONDecodeError:
                history = []
    
    history.append(entry)
    
    with open(master_file, "w") as f:
        json.dump(history, f, indent=4)
    print(f"Appended entry to {master_file}")

In [14]:
# 1. Define your prompt (ensure you mention JSON!)
messages = [
    {"role": "system", "content": "You are a helpful assistant. Always output valid JSON."},
    {"role": "user", "content": "Generate a profile for a fictional space explorer. Include 'name', 'rank', and 'home_planet'."}
]

# 2. Call the LLM
content, stats = call_llm_json(messages)

# 3. Log and Save
if content:
    # Print it to the screen
    print(f"Success! Model used: {stats['model']}")
    
    # Save #1: Individual file
    save_single_json(content, "explorer_profile.json")
    
    # Save #2: Master log with telemetry
    append_to_master_json(content, stats)
else:
    print(f"Failed: {stats['error']}")

Success! Model used: gpt-4o
Saved snapshot to explorer_profile.json
Appended entry to master_history.json
